In [1]:
import torch
from layers import activation
import random
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms import ToTensor
import numpy as np
import pywt
import pickle

SyntaxError: 'return' outside function (layers.py, line 109)

In [17]:
#### WAVELET Neural Operator ####
class SWTLayer(nn.Module):
    # M: grid discretizaiton
    # N: output dimension
    # N2: dimension of lifted output
    def __init__(self, M, Nlifted, kmax, wavelet='db1', level=None): 
        super().__init__()
        if M <= 0: raise ValueError("in_features must be a positive integer")  
        if bin(M).count('1') != 1: raise ValueError("truncate at a wavelet stage")

         # wavelet transform specs
        self.wavelet = wavelet
        if not level: self.level = pywt.dwt_max_level(M, pywt.Wavelet(wavelet).dec_len)
        if kmax > self.level: raise ValueError("pick lower stage to truncate at")  
        self.slices = np.arange(0, (self.level+1)*M + M, M) # store the indices of the wavelet levels

        self.weights = nn.Parameter(torch.rand((Nlifted, Nlifted, M*(self.level+1))))
        self.bias = nn.Parameter(torch.randn((Nlifted, Nlifted)))
        self.kmax = kmax*M # stages to truncate

    # Fourier transform, ready input for convolution 
    def data_transform(self, x):     
        x_transformed = np.concatenate(
                            pywt.swt(
                                x.detach().numpy(), 
                                self.wavelet, 
                                level=self.level, 
                                axis=1, 
                                trim_approx=True
                            ), 
                        axis=1) # returns transform as a list of coefficients
        
        x_transformed[:, self.kmax:, :] = 0 # truncate high frequencies
        
        return torch.tensor(x_transformed)

    # Transform back into physical space
    def inverse_tranform(self, x_transformed):      
        # cast into form compatible with pywt.waverec (list of arrays)
        x_transformed = [x_transformed[:, self.slices[i]:self.slices[i+1], :].detach().numpy() for i in range(len(self.slices)-1)]

        # return inverse transform
        return torch.tensor(pywt.iswt(x_transformed, wavelet=self.wavelet, axis=1))
        
    def forward (self, x):
        # Fourier transform
        x_transformed = self.data_transform(x)

        # Linear convolution
        x_transformed = torch.einsum("bkw, wmk -> bkm", x_transformed, self.weights)

        # Inverse transform + bias 
        x_out = self.inverse_tranform(x_transformed) + nn.functional.linear(x, self.bias, bias=None)
        return  x_out 

In [18]:
class DWTLayer(nn.Module):
    # M: grid discretizaiton
    # N: output dimension
    # N2: dimension of lifted output
    def __init__(self, M, Nlifted, kmax, wavelet='db1', level=None): 
        super().__init__()
        
        if M <= 0: raise ValueError("in_features must be a positive integer")  
        if bin(M//kmax).count('1') != 1: raise ValueError("truncate at a wavelet stage")
            
        self.weights = nn.Parameter(torch.rand((Nlifted, Nlifted, M)))
        self.bias = nn.Parameter(torch.randn((Nlifted, Nlifted)))
        self.kmax = kmax # stages to truncate

        # wavelet transform specs
        self.wavelet = wavelet
        if not level: self.level = pywt.dwt_max_level(M, pywt.Wavelet(wavelet).dec_len)
        coeffs = pywt.wavedec(np.random.randn(M), self.wavelet, level=self.level) # store the indices of the wavelet levels
        slices = [len(c) for c in coeffs]
        slices = [sum(slices[:i]) for i in range(len(slices))] + [M]
        self.slices = slices

    # Fourier transform, ready input for convolution
    def data_transform(self, x):     
        x_transformed = np.concatenate(
                            pywt.wavedec(
                                x.detach().numpy(), 
                                self.wavelet, 
                                level=self.level, 
                                axis=1
                            ), 
                        axis=1) # returns transform as a list of coefficients
        
        x_transformed[:, self.kmax:, :] = 0 # truncate high frequencies
        
        return torch.tensor(x_transformed)

    # Transform back into physical space
    def inverse_tranform(self, x_transformed):      
        # cast into form compatible with pywt.waverec (list of arrays)
        x_transformed = [x_transformed[:, self.slices[i]:self.slices[i+1], :].detach().numpy() for i in range(len(self.slices)-1)]
        
        return torch.tensor(pywt.waverec(x_transformed, wavelet=self.wavelet, axis=1))
        
    def forward (self, x):
        # Fourier transform
        x_transformed = self.data_transform(x)

        # Linear convolution
        x_transformed = torch.einsum("bkw, wmk -> bkm", x_transformed, self.weights)

        # Inverse transform + bias 
        x_out = self.inverse_tranform(x_transformed) + nn.functional.linear(x, self.bias, bias=None)
        return  x_out 


In [88]:
class nFourierLayer(nn.Module):
    # M: grid discretizaiton
    # N: output dimension
    # N2: dimension of lifted output
    def __init__(self, dimensions, Nlifted, kmax, transform_type="fourier", wavelet='db1', level=None): 
        super().__init__()

        if len(kmax) != len(dimensions)-1: 
            raise ValueError('Please provide truncation wavenumbers for all dimensions')
        self.dimensions = dimensions
        self.axes = tuple(range(1, len(dimensions), 1))
        self.weights = nn.Parameter(torch.rand((Nlifted, Nlifted)+ dimensions[:-2] + (dimensions[-2]//2+1,), dtype=torch.cfloat)) 
        self.bias = nn.Parameter(torch.randn((Nlifted, Nlifted)))      
        
        # indices to truncate
        self.k_to_truncate = tuple([list(range(kmax[i], dimensions[i]-kmax[i])) for i in range(len(kmax)-1)] )
        self.k_to_truncate += tuple([list(range(kmax[-1], dimensions[-2]//2+1))])

        # Einstein summation code
        einsum_alphabet = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
        einsum_operand_a = einsum_alphabet[0]
        for i in range(len(dimensions)):
            einsum_operand_a += einsum_alphabet[i+1] # code for operand a  
        einsum_operand_b = einsum_operand_a[-1] + einsum_alphabet[len(dimensions)+1] + einsum_operand_a[1:-1] 
        einsum_output = einsum_operand_a[:-1] + einsum_operand_b[1]
        self.einsum_indices = einsum_operand_a + "," + einsum_operand_b + "->" + einsum_output

        # choose data transforms (fourier or wavelet)
    #    if transform_type == "fourier":
    #        self.data_transform = self.fourier_transform
    #        self.inverse_transform = self.inverse_fourier_transform
    #    elif transform_type == "wavelet":
    #        if any( bin(dimensions[i]//kmax[i]).count('1') != 1  for i in range(len(kmax))): 
    #            raise ValueError("must choose a power of 2 at which to truncate")
    #        self.wavelet = wavelet
    #        self.level = []
    #        self.slices = []
    #        if not level: 
    #            self.level += [ pywt.dwt_max_level(d, pywt.Wavelet(wavelet).dec_len) for d in dimensions[:-1] ]
    #            coeffs = [
    #                pywt.wavedec(np.random.randn(dimensions[i]), self.wavelet, level=self.level[i]) 
    #                for i in range(len(dimensions)-1) 
    #                    ] # store the indices of the wavelet levels
    #            slices = [[len(c) for c in coeff] for coeff in coeffs]
    #            slices = [[sum(slices[j][:i]) for i in range(len(slices[j][:]))] + [dimensions[j]] for j in range(len(dimensions)-1) ]
    #            self.slices = slices
    #        self.data_transform = self.wavelet_transform
    #        self.inverse_transform = self.inverse_wavelet_transform
    #    else:
    #        raise ValueError("Please pick 'fourier' or 'wavelet' as your transform type")

    # Fourier transform, ready input for convolution
    def fourier_transform(self, x):
        return torch.fft.rfftn(x, dim=self.axes, norm="ortho")

    # Transform back into physical space
    def inverse_fourier_transform(self, x_transformed):
        return torch.fft.irfftn(x_transformed, dim=self.axes,  norm="ortho")

    #def wavelet_transform(self, x):  
        #for i in range(len(self.axes)):
        #    x_transformed = torch.tensor(np.concatenate(
        #                    pywt.wavedec(
        #                        x.detach().numpy(), 
        #                        self.wavelet, 
        #                        level=self.level[i], 
        #                        axis=self.axes[i]
        #                    ), 
        #                axis=self.axes[i])) # returns transform as a list of coefficients   
   #     pywt.wavedecn(
        
   #     return x_transformed

    # Transform back into physical space
   # def inverse_wavelet_transform(self, x_transformed):      
   #     # cast into form compatible with pywt.waverec (list of arrays)
   #     x_transformed = [x_transformed[:, self.slices[i]:self.slices[i+1], :].detach().numpy() for i in range(len(self.slices)-1)]
   #     for i in range(len(self.axes)):
   #         
   #     #torch.tensor(pywt.waverec(x_transformed, wavelet=self.wavelet, axis=1))
   #     return 

    def truncate(self, x_transformed):
        x_transformed[:, np.ix_(*self.k_to_truncate)] = 0
        return x_transformed
        
    def forward (self, x):
        # Fourier transform
        x_transformed = self.data_transform(x)

        # Truncate
        x_transformed = self.truncate(x_transformed)

        # Linear convolution
        x_transformed = torch.einsum(self.einsum_indices, x_transformed, self.weights)

        # Inverse transform + bias 
        x_out = self.inverse_tranform(x_transformed) + nn.functional.linear(x, self.bias, bias=None)
        return  x_out 

In [89]:
class nFNO(nn.Module):
    # dimensions: tuple with dimension of each element in batch
    # N2: lifted dimension output
    # k_truncate: tuple with max wavenumbers beyond which to truncate
    # depth: layer depth
    def __init__(self, dimensions, N2, k_truncate, depth=1, transform_type="fourier", wavelet="db1", activation_fun=nn.functional.relu):
        super().__init__()
        # lifting layer
        initial_layer = [nn.Linear(dimensions[-1], N2)]

        # fourier layer with batch normalization
        fourier_layer = [nFourierLayer(dimensions, N2, k_truncate, transform_type, wavelet), 
                         nn.BatchNorm1d(dimensions[0]),
                         activation(activation_fun)]

        # output layer ("decoder")
        final_layer = [nn.Linear(N2, dimensions[-1])]
        
        layer_list = initial_layer
        for _ in range(depth): 
            layer_list += fourier_layer 
        layer_list += final_layer
        
        self.stack = nn.Sequential(*layer_list)      
                
    def forward(self, x):
        return self.stack(x)


In [90]:

# Define the Fourier Neural Operator
class WNO(nn.Module):
    # M: grid discretizaiton
    # N: output dimension
    # N2: lifted output
    def __init__(self, M, N, N2, Mk_truncate, depth=1, wavelet='db1'):   
        super().__init__()
        # lifting layer
        initial_layer = [nn.Linear(N, N2)]

        # fourier layer with batch normalization
        fourier_layer = [SWTLayer(M, N2, Mk_truncate, wavelet='db1', level=None), 
                         nn.BatchNorm1d(M),
                         activation()]

        # output layer ("decoder")
        final_layer = [nn.Linear(N2, N)]
        
        layer_list = initial_layer
        for _ in range(depth): 
            layer_list += fourier_layer 
        layer_list += final_layer
        
        self.stack = nn.Sequential(*layer_list)      
                
    def forward(self, x):
        return self.stack(x)

In [91]:
# Load PDE solution
with open('data/KuramotoSivashinsky/KS_N32.pkl', 'rb') as file:
    data = pickle.load(file)
data = torch.tensor(data)
Nx = data.size()[1]
print(data.size())

torch.Size([2001, 32])


In [92]:
# Make datasets, dataloaders
history = 1
shuffle_flag=True # shuffling helps a lot
dt = 20 # increasing this erodes prediction accuracy
training_size = 1024
data_truncated = data[:history*(data.size()[0]//history), :]

features = data_truncated[:-dt, :].to(torch.float32)
features = torch.reshape(features, (features.size()[0], features.size()[1], 1))

labels = data_truncated[dt:, :].to(torch.float32)
labels = torch.reshape(labels, (labels.size()[0], labels.size()[1], 1))

# datasets
if shuffle_flag: 
    idx = torch.randperm(training_size)
else:
    idx = torch.arange(training_size)
training_dataset = TensorDataset(features[idx, :, :], labels[idx, :])
testing_dataset = TensorDataset(features[training_size:, :, :], labels[training_size:, :])

# dataloaders
batch_size = 64
training_dataloader = DataLoader(training_dataset, batch_size=batch_size, shuffle=False)
testing_dataloader = DataLoader(testing_dataset, batch_size=batch_size, shuffle=False)

In [93]:
# initialize operator
device = "cpu"
Nf = 1
Nlifted = 16
Nk_truncated = 8
depth = 1
neuralop = nFNO((Nx, Nf), Nlifted, (Nk_truncated,), depth, transform_type="fourier").to(device) # lifted dim: not too small, not too large # fourier layer: not too deep
loss_fn = torch.nn.MSELoss(reduction='mean')
rate = 2E-04
optimizer_neuralop = torch.optim.Adam(neuralop.parameters(), lr=rate)
progress = []

In [71]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)     
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

def test(dataloader, model, loss_fn):
    num_batches = len(dataloader)
    model.eval()
    loss = 0
    with torch.no_grad():   
        for batch, (X,y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss += loss_fn(pred, y).item()
    loss /= num_batches
    return loss

In [72]:
# train
epochs = 600
for i in range(epochs):
    train(training_dataloader, neuralop, loss_fn, optimizer_neuralop)
    progress += [test(testing_dataloader, neuralop, loss_fn)]
    if i%100==0:
        print(f"Loss: {(progress[-1]):>0.5f}")
        rate *= 0.5
        optimizer_neuralop = torch.optim.Adam(neuralop.parameters(), lr=rate)
    if np.isnan(progress[-1]): 
        print('nan!')
        break
print(f'final learning rate: {rate}')
plt.figure()
plt.plot(progress)
plt.loglog()

TypeError: '<' not supported between instances of 'list' and 'int'

In [117]:
x = np.array(
    [[1, 2,   3, 4,   5], 
     [9, 10, 11, 12, 13]]
)
idx = [[3,4]]

x[:, np.ix_(*idx)] = 0
print(x)

[[ 1  2  3  0  0]
 [ 9 10 11  0  0]]


In [120]:
4//2 + 1

3

In [147]:
class H1Loss(nn.Module):
    def __init__(self, reduction="mean"):
        super().__init__()
        if reduction not in ("mean", "sum", "none"):
            raise ValueError(f"Invalid reduction: {reduction}")
        self.reduction = reduction

    # compute first order derivative
    def diff(self, y):
        for i in range(1, len(y.shape)-1):
            y = roll(y, shifts=1, dims=i) - y
        return y
        
    def forward(self, yin, ytarget):
        loss = (yin - ytarget) ** 2 + (self.diff(yin) - self.diff(ytarget))**2

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:  # 'none'
            return loss

In [148]:
loss = H1Loss()

In [150]:
loss(torch.rand((2, 3, 1)), torch.rand((2, 3, 1)))

hi
hi


tensor(0.7605)

In [76]:
a = torch.rand((3, 2))
print(a)
print(torch.mean(a, dim=list(range(1, a.dim()))))

tensor([[0.3949, 0.8575],
        [0.4290, 0.4622],
        [0.6786, 0.3722]])
tensor([0.6262, 0.4456, 0.5254])


In [77]:
a - torch.mean(a, dim=list(range(1, a.dim())), keepdim=True)

tensor([[-0.2313,  0.2313],
        [-0.0166,  0.0166],
        [ 0.1532, -0.1532]])

In [81]:
0.4290 - 0.4456

-0.016600000000000004

In [66]:
x_transformed = torch.rand((10, 4, 3, 4))
interior = [slice(None,)]
zero_mode = [slice(None,)]
for i in range(1, x_transformed.ndim-1):
    interior += [slice(1, None)]
    zero_mode += [0]
print(x_transformed[tuple(interior)].shape)
print(torch.reshape(x_transformed[tuple(zero_mode)], [10, 1, 1, 4]).shape)
torch.cat((torch.reshape(x_transformed[tuple(zero_mode)], [10, 1, 1, 4]), x_transformed[tuple(interior)]), 1).shape

torch.Size([10, 3, 2, 4])
torch.Size([10, 1, 1, 4])


RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 1 but got size 2 for tensor number 1 in the list.

In [59]:
zero_mode

[slice(None, None, None), (0,), (0,)]